## Environment Setup
Do not modify the following two cells. adh_init() details here: [README-adh_init](https://github.com/cfacorp/aao-dbx-helpers/blob/master/README-adh_init.md).

In [0]:
%run /Workspace/Repos/global/aao-dbx-helpers/.boot/adh_init

In [0]:
dbutils.widgets.removeAll()

In [0]:
__adh_init()

## Task Code

In [0]:
import pandas as pd
from redshift import load_from_redshift
import mlflow
from mlflow.tracking import MlflowClient
from constants import ANALOG_MODEL_NAME

In [0]:
CATALOG_NAME = spark.conf.get("cfa.default_catalog")
SCHEMA_NAME = spark.conf.get("cfa.default_schema")
print(f"{CATALOG_NAME} {SCHEMA_NAME=}")

In [0]:
projected_annual_sales_growth_query = """
SELECT 
    sales_lower_bound, 
    sales_upper_bound, 
    sales_growth_rate 
FROM 
    rd_datamart.conf.projected_annual_sales_growth 
WHERE 
    1=1 
    AND valid_from <= current_date 
    AND valid_to >= current_date
"""
projected_annual_sales_growth = load_from_redshift(
    projected_annual_sales_growth_query
).toPandas()
projected_annual_sales_growth.shape

In [0]:
location_attributes_query = f"""
SELECT 
    location_num,
    location_name,
    latitude,
    longitude,
    subtype,
    location_type,
    location_status,
    state,
    dma,
    submarket,
    dma_id,
    region,
    opened_dt,
    ssam_flag,
    no_dt_flag,
    total_population_1,
    worker_population_1,
    daytime_population_1,
    avg_population_growth_1,
    total_households_1,
    total_households_per_sq_mile_1,
    density_classification_1,
    med_household_income_1,
    med_disposable_income_1,
    avg_disposable_income_1,
    annual_budget_expenditure_1,
    annual_budget_expenditure_index_1,
    med_home_value_1,
    avg_home_value_1,
    food_away_from_home_1,
    competitor_count_1,
    total_population_3,
    worker_population_3,
    daytime_population_3,
    avg_population_growth_3,
    total_households_3,
    total_households_per_sq_mile_3,
    density_classification_3,
    med_household_income_3,
    med_disposable_income_3,
    avg_disposable_income_3,
    annual_budget_expenditure_3,
    annual_budget_expenditure_index_3,
    med_home_value_3,
    avg_home_value_3,
    food_away_from_home_3,
    competitor_count_3,
    total_population_5,
    worker_population_5,
    daytime_population_5,
    avg_population_growth_5,
    total_households_5,
    total_households_per_sq_mile_5,
    density_classification_5,
    med_household_income_5,
    med_disposable_income_5,
    avg_disposable_income_5,
    annual_budget_expenditure_5,
    annual_budget_expenditure_index_5,
    med_home_value_5,
    avg_home_value_5,
    food_away_from_home_5,
    competitor_count_5
FROM
    {CATALOG_NAME}.{SCHEMA_NAME}.location_features
WHERE
    location_type in ('existing', 'pipeline')
"""
location_attributes_df = spark.sql(location_attributes_query)
location_attributes_pdf = location_attributes_df.toPandas()
location_attributes_pdf.shape

In [0]:
boundary_features_query = f"""
SELECT 
    location_num,
    boundary_5,
    boundary_10,
    boundary_15,
    boundary_20,
    boundary_25,
    boundary_30,
    months_6m,
    months_12m,
    months_18m,
    months_24m,
    months_30m,
    months_36m,
    avg_monthly_sales_closest_1year,
    avg_monthly_sales_mean_1year,
    avg_monthly_sales_25th_1year,
    avg_monthly_sales_75th_1year,
    avg_monthly_sales_closest_6month,
    avg_monthly_sales_mean_6month,
    avg_monthly_sales_25th_6month,
    avg_monthly_sales_75th_6month,
    growth_rate_closest,
    growth_rate_mean,
    growth_rate_25th,
    growth_rate_75th
FROM 
    {CATALOG_NAME}.{SCHEMA_NAME}.boundary_features
"""
boundary_features_df = spark.sql(boundary_features_query)
boundary_features_pdf = boundary_features_df.toPandas()
boundary_features_pdf.shape

In [0]:
hexbin_level_features_query = f"""
SELECT 
    location_num,
    bigbox_1,
    bigbox_2,
    bigbox_3,
    competitor_1,
    competitor_2,
    competitor_3,
    highway_5,
    highway_10,
    highway_5min,
    highway_10min,
    close_highway_dist,
    close_highway_min,
    close_ramp_1,
    close_ramp_2,
    interstate_within_2min,
    median_average_daily_traffic_1,
    max_average_daily_traffic_1,
    median_average_daily_traffic_2,
    max_average_daily_traffic_2,
    median_average_daily_traffic_3,
    max_average_daily_traffic_3
FROM 
    {CATALOG_NAME}.{SCHEMA_NAME}.hexbin_level_features
"""
hexbin_level_features_df = spark.sql(hexbin_level_features_query)
hexbin_level_features_pdf = hexbin_level_features_df.toPandas()
hexbin_level_features_pdf.shape

In [0]:
final_features_df = location_attributes_df.join(boundary_features_df, "location_num", "inner") \
    .join(hexbin_level_features_df, "location_num", "inner") \
    .drop(boundary_features_df["location_num"], hexbin_level_features_df["location_num"]).toPandas()

In [0]:
monthly_sales_query = f"SELECT * FROM {CATALOG_NAME}.{SCHEMA_NAME}.monthly_sales"
monthly_sales_pdf = spark.sql(monthly_sales_query).toPandas()
monthly_sales_pdf["month"] = pd.to_datetime(monthly_sales_pdf["month"])
monthly_sales_pdf = monthly_sales_pdf.dropna(
    subset=["location_num", "month", "monthly_sales"]
)
monthly_sales_pdf.shape

In [0]:
# The following commented imports are helpful for interative development
# eliminating the need to detach/reattach and re-run all imports

# from importlib import reload
# import shared
# reload(shared)
# import shared.analog_model
# reload(shared.analog_model)

from shared.analog_model import AnalogStoresModel

In [0]:

def write_to_disk_and_load(df: pd.DataFrame, path: str) -> pd.DataFrame:
    """
    Writes a DataFrame to disk in Parquet format and then loads it back. This is done to manage data type difference between Pandas and Spark dataframes.

    Parameters:
    df (pd.DataFrame): The DataFrame to be written to disk.
    path (str): The file path where the DataFrame will be saved.

    Returns:
    pd.DataFrame: The DataFrame loaded from the saved Parquet file.
    """
    df.to_parquet(path)
    df_new = pd.read_parquet(path)
    return df_new


In [0]:
run_mode = dbutils.widgets.get("run_mode")
if run_mode == "local":
    local_flag = True
else:
    local_flag = False

print(local_flag)
model = AnalogStoresModel(ANALOG_MODEL_NAME)
location_attributes_pdf["opened_dt"] = location_attributes_pdf["opened_dt"].astype(str)

DEFAULT_CATALOG = adh.conf.default_schema_qualified.split(".")[0]
DEFAULT_SCHEMA = adh.conf.default_schema_qualified.split(".")[1]

# Create volume for parquet files that fix data type issue between Pandas and Spark dataframes
spark.sql(
    f"""
    CREATE VOLUME IF NOT EXISTS {DEFAULT_CATALOG}.{DEFAULT_SCHEMA}.mlflow_float_fix
"""
)

# Create volume for parquet files saved as artifacts in MLFlow model
artifacts_volume_name = "mlflow_artifacts"
spark.sql(
    f"""
    CREATE VOLUME IF NOT EXISTS {DEFAULT_CATALOG}.{DEFAULT_SCHEMA}.{artifacts_volume_name}
"""
)

base_volume_path = f"/Volumes/{DEFAULT_CATALOG}/{DEFAULT_SCHEMA}"

float_fix_path = f"{base_volume_path}/mlflow_float_fix"

# This fixes a decimal serialization issue in MLflow

location_attributes_path = f"{float_fix_path}/location_attributes.parquet"
location_attributes_pdf_new = write_to_disk_and_load(
    location_attributes_pdf, location_attributes_path
)

boundary_features_path = f"{float_fix_path}/boundary_features_df.parquet"
boundary_features_pdf_new = write_to_disk_and_load(
    boundary_features_pdf, boundary_features_path
)

hexbin_level_features_path = f"{float_fix_path}/hexbin_level_features_df.parquet"
hexbin_level_features_pdf_new = write_to_disk_and_load(
    hexbin_level_features_pdf, hexbin_level_features_path
)

monthly_sales_path = f"{float_fix_path}/monthly_sales_df.parquet"
monthly_sales_pdf_new = write_to_disk_and_load(monthly_sales_pdf, monthly_sales_path)

projected_annual_sales_growth_path = (
    f"{float_fix_path}/projected_annual_sales_growth_df.parquet"
)
projected_annual_sales_growth_new = write_to_disk_and_load(
    projected_annual_sales_growth, projected_annual_sales_growth_path
)

model.fit(
    location_attributes_pdf_new,
    boundary_features_pdf_new,
    hexbin_level_features_pdf_new,
    monthly_sales_pdf_new,
    projected_annual_sales_growth_new,
)
model.log_model(
    run_id=adh.params.run_id,
    schema=adh.conf.default_schema_qualified,
    artifacts_volume_name=artifacts_volume_name,
    env=adh.params.environment,
    local=local_flag,
)

In [0]:

# Load the model
if local_flag:
    model_path = ANALOG_MODEL_NAME
else:
    client = MlflowClient()
    model_name = f"{adh.conf.default_schema_qualified}.{ANALOG_MODEL_NAME}"
    print(model_name)
    latest_version = max(
        [int(v.version) for v in client.search_model_versions(f"name = '{model_name}'")]
    )
    model_path = f"models:/{model_name}/{latest_version}"

model_retrieved = mlflow.pyfunc.load_model(model_path)

python_model_obj = model_retrieved._model_impl.python_model

# Use the model
all_analogs = []
skipped_stores = []
threshold = 200
point_stores = python_model_obj.final_features_df[
    python_model_obj.final_features_df.location_status_x.isin(["future"])
]
for i, row in point_stores.iterrows():
    loc = row["location_num"]

    print(f"Finding analog stores for loc: {loc} with status: {row.location_status_x}")
    query = f"""
    SELECT
        origin_id AS origin_location_number,
        destination_id AS destination_location_number,
        total_distance_miles AS total_travel_distance
    FROM 
        rd_datamart.drive_time.driving_location_routing_matrix
    WHERE 
        total_travel_distance <= {threshold}
        AND origin_location_number = '{loc}'
        AND destination_site_status = 'opened'
    ORDER BY 
        total_travel_distance
    LIMIT 50
    """
    distances = load_from_redshift(query).toPandas()
    if len(distances) == 0:
        print(f"No analogs found for {loc} with status: {row.location_status_x}")
        skipped_stores.append(loc)
    else:
        store_features = pd.DataFrame(row).transpose().reset_index()
        # The predict function only accepts a dataframe or a series and not custom parameters. To get around this we pass function parameters as a cell, we serialize the dataframe and pass it as a string
        model_input = pd.DataFrame(
            [
                {
                    "k": 10,
                    "store_features": python_model_obj.serialize_df(store_features),
                    "distances": python_model_obj.serialize_df(distances),
                }
            ]
        )
        model_input["k"] = model_input["k"].astype("int32")
        analogs = model_retrieved.predict(model_input)
        analogs["origin_location_status"] = row.location_status_x
        all_analogs.append(analogs)
print("All skipped stores:", skipped_stores)
analogs_df = pd.concat(all_analogs).reset_index(drop=True)
analogs_df["distance_gt_200"] = analogs_df["distance"].apply(
    lambda x: x == ">200miles" or x > 200
)
analogs_df["distance"] = pd.to_numeric(analogs_df["distance"], errors="coerce")

In [0]:
print("All skipped stores:", skipped_stores)
analogs_df = pd.concat(all_analogs).reset_index(drop=True)
analogs_df["distance_gt_200"] = analogs_df["distance"].apply(
    lambda x: x == ">200miles" or x > 200
)
analogs_df["distance"] = pd.to_numeric(analogs_df["distance"], errors="coerce")

In [0]:
display(analogs_df)

In [0]:
# Convert pandas dataframe to spark dataframe
spark_analogs_df = spark.createDataFrame(analogs_df)

# Write the spark dataframe to the delta table, overwriting the existing and schema each time
(
    spark_analogs_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{adh.conf.default_schema_qualified}.analog_stores")
)

Validation checks

In [0]:
distinct_count = (
    spark_analogs_df.groupBy("origin_location_number").count().where("count < 5")
)

In [0]:
display(distinct_count)

In [0]:
display(
    boundary_features_df.join(
        distinct_count,
        boundary_features_df["location_num"] == distinct_count["origin_location_number"],
        "inner",
    ).orderBy("count")
)

In [0]:
display(
    hexbin_level_features_df.join(
        distinct_count,
        hexbin_level_features_df["location_num"]
        == distinct_count["origin_location_number"],
        "inner",
    ).orderBy("count")
)

In [0]:
state_exception = (
    location_attributes_df.join(
        distinct_count,
        location_attributes_df["location_num"] == distinct_count["origin_location_number"],
        "inner",
    )
    .orderBy("count")
    .where("state NOT IN ('MT', 'HI', 'AK', 'ND', 'SD')")
    .count()
)
print( f"{state_exception == 0} All locations should have at least 5 analogs")